In [ ]:
# Rutas portables: ejecutar desde notebooks/ o desde la raíz clonada.
from pathlib import Path
import sys, os
_repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs/pipeline.json').exists())
sys.path.insert(0, str(_repo / 'src'))
from pipeline_runtime import prepare_runtime
_runtime = prepare_runtime(_repo)
os.chdir(_runtime)
sys.path.insert(0, str(_runtime))

import json
_central = json.loads((_repo / 'configs/pipeline.json').read_text(encoding='utf-8'))


# Fase 4.0 — Selección multivariada por correlación WoE

Se consumen artefactos B.3 congelados. La selección utiliza exclusivamente TRAIN y prioriza IV final. No se entrena un modelo predictivo. Ejecutar las celdas en orden desde la raíz del proyecto. Cambiar únicamente `CORR_THRESHOLD` para ajustar el umbral.

## MULT.1 — Importación y configuración

**Qué hace:** Importación y configuración.

**Recibe:** Módulos 3.0/4.0 y rutas existentes.

**Produce:** biv, mult, rutas y CORR_THRESHOLD.

**Funciones:** 4.0.0.multivariado.py: import_bivariado.

In [ ]:
# MULT.1: Importación y configuración
# Funciones: 4.0.0.multivariado.py: import_bivariado
from pathlib import Path
import importlib.util
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
spec = importlib.util.spec_from_file_location('multivariado', '4.0.0.multivariado.py')
mult = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mult)
biv = mult.import_bivariado()
CORR_THRESHOLD = 0.60
BIV_ROOT = Path('4.ANALIS_UV_BV_MV/BIVARIADO')
MULT_ROOT = Path('4.ANALIS_UV_BV_MV/MULTIVARIADO')
BINNING_PATH = BIV_ROOT / 'binning_woe_train.json'
pd.set_option('display.max_rows', 100)
print('Pearson TRAIN; descarte estricto abs(r) >', CORR_THRESHOLD)
print('Empate IV: nombre alfabético. Referencia: máximo |r|; empate por orden IV.')
CORR_THRESHOLD = _central["correlation"]["threshold"]


## MULT.2 — Carga de muestras v4

**Qué hace:** Carga de muestras v4.

**Recibe:** Artefactos v4 existentes.

**Produce:** muestras_biv, X originales, targets y validación.

**Funciones:** 3.0.0.bivariado.py: load_v4; 4.0.0.multivariado.py: validate_inputs.

In [ ]:
# MULT.2: Carga de muestras v4
# Funciones: 3.0.0.bivariado.py: load_v4; 4.0.0.multivariado.py: validate_inputs
muestras_biv = biv.load_v4()
X_train_v4, X_test_v4, X_oot_v4 = [muestras_biv['X'][s] for s in mult.SEGMENTS]
y_train, y_test, y_oot = [muestras_biv['y'][s] for s in mult.SEGMENTS]
df_validacion_entradas = mult.validate_inputs(muestras_biv)
display(df_validacion_entradas)
targets_originales = {s: muestras_biv['y'][s].copy(deep=True) for s in mult.SEGMENTS}

## MULT.3 — Carga de cortes WoE y transformación

**Qué hace:** Carga de cortes WoE y transformación.

**Recibe:** JSON B.3, originales v4 y CSV WoE B.3.

**Produce:** X_train_woe, X_test_woe, X_oot_woe y diagnósticos.

**Funciones:** 3.0.0.bivariado.py: load_binning, transform_woe; 4.0.0.multivariado.py: frozen_transform.

In [ ]:
# MULT.3: Carga de cortes WoE y transformación
# Funciones: 3.0.0.bivariado.py: load_binning, transform_woe; 4.0.0.multivariado.py: frozen_transform
binning_woe_train = biv.load_binning(BINNING_PATH)
binning_hash_inicial = mult.sha256(BINNING_PATH)
matrices_woe, df_diagnosticos_woe = mult.frozen_transform(
    muestras_biv, binning_woe_train, biv, BIV_ROOT / 'matrices')
X_train_woe, X_test_woe, X_oot_woe = [matrices_woe[s] for s in mult.SEGMENTS]
assert len(binning_woe_train['approved_variables']) == 24, 'La referencia B.3 esperada cambió; revisar antes de continuar'
display(df_diagnosticos_woe)
print('Transformación exacta frente a CSV B.3; aprobadas:', X_train_woe.shape[1])

## MULT.4 — Carga de IV y ranking

**Qué hace:** Carga de IV y ranking.

**Recibe:** Resumen B.3 final y artefacto congelado.

**Produce:** df_iv_final, cols_por_iv.

**Funciones:** 4.0.0.multivariado.py: rank_iv.

In [ ]:
# MULT.4: Carga de IV y ranking
# Funciones: 4.0.0.multivariado.py: rank_iv
df_resumen_bivariado = pd.read_csv(BIV_ROOT / 'reportes/df_resumen_bivariado.csv', float_precision='round_trip')
df_iv_final = mult.rank_iv(df_resumen_bivariado, binning_woe_train, X_train_woe.columns)
cols_por_iv = df_iv_final.Variable.tolist()
display(df_iv_final)

## MULT.5 — Matriz de correlación Pearson TRAIN

**Qué hace:** Matriz de correlación Pearson TRAIN.

**Recibe:** X_train_woe y cols_por_iv; no recibe TEST/OOT.

**Produce:** corr_woe y corr_woe_absoluta.

**Funciones:** 4.0.0.multivariado.py: train_correlation.

In [ ]:
# MULT.5: Matriz de correlación Pearson TRAIN
# Funciones: 4.0.0.multivariado.py: train_correlation
corr_woe = mult.train_correlation(X_train_woe, cols_por_iv)
corr_woe_absoluta = corr_woe.abs()
display(corr_woe)
display(corr_woe_absoluta)

## MULT.6 — Heatmap completo WoE

**Qué hace:** Heatmap completo WoE.

**Recibe:** corr_woe en orden IV.

**Produce:** fig_heatmap_completo.

**Funciones:** 4.0.0.multivariado.py: heatmap.

In [ ]:
# MULT.6: Heatmap completo WoE
# Funciones: 4.0.0.multivariado.py: heatmap
fig_heatmap_completo = mult.heatmap(corr_woe, 'Correlación Pearson en espacio WoE — TRAIN')
plt.show()

## MULT.7 — Selección por correlación con prioridad IV

**Qué hace:** Selección por correlación con prioridad IV.

**Recibe:** df_iv_final, corr_woe TRAIN y CORR_THRESHOLD.

**Produce:** variables_finales, df_correlacion_decisiones, df_correlaciones_descartadas.

**Funciones:** 4.0.0.multivariado.py: select_sequential.

In [ ]:
# MULT.7: Selección por correlación con prioridad IV
# Funciones: 4.0.0.multivariado.py: select_sequential
variables_finales, df_correlacion_decisiones, df_correlaciones_descartadas = mult.select_sequential(
    df_iv_final, corr_woe, CORR_THRESHOLD)
print('Iniciales:', len(cols_por_iv), '| Conservadas:', len(variables_finales),
      '| Eliminadas:', len(cols_por_iv) - len(variables_finales))

## MULT.8 — Tabla de variables conservadas y descartadas

**Qué hace:** Tabla de variables conservadas y descartadas.

**Recibe:** Decisiones de MULT.7.

**Produce:** Tablas completas y detalle de todas las referencias.

**Funciones:** 4.0.0.multivariado.py: resultados de select_sequential.

In [ ]:
# MULT.8: Tabla de variables conservadas y descartadas
# Funciones: 4.0.0.multivariado.py: resultados de select_sequential
display(df_correlacion_decisiones)
display(df_correlaciones_descartadas)
display(df_correlacion_decisiones.groupby('Estado_correlacion').size().rename('Variables').to_frame())

## MULT.9 — Heatmap final

**Qué hace:** Heatmap final.

**Recibe:** corr_woe y variables_finales.

**Produce:** corr_woe_final y fig_heatmap_final.

**Funciones:** 4.0.0.multivariado.py: heatmap.

In [ ]:
# MULT.9: Heatmap final
# Funciones: 4.0.0.multivariado.py: heatmap
corr_woe_final = corr_woe.loc[variables_finales, variables_finales]
pares_finales = corr_woe_final.abs().to_numpy()[np.triu_indices(len(variables_finales), 1)]
assert not (pares_finales > CORR_THRESHOLD).any()
print('Máximo |r| final:', float(pares_finales.max()) if len(pares_finales) else 'No hay parejas')
fig_heatmap_final = mult.heatmap(corr_woe_final, 'Correlación Pearson WoE — finalistas TRAIN')
plt.show()

## MULT.10 — Ranking IV y decisiones

**Qué hace:** Ranking IV y decisiones.

**Recibe:** df_correlacion_decisiones.

**Produce:** fig_ranking_iv.

**Funciones:** 4.0.0.multivariado.py: plot_iv.

In [ ]:
# MULT.10: Ranking IV y decisiones
# Funciones: 4.0.0.multivariado.py: plot_iv
fig_ranking_iv = mult.plot_iv(df_correlacion_decisiones)
plt.show()

## MULT.11 — Aplicación de variables finales a TEST/OOT

**Qué hace:** Aplicación de variables finales a TEST/OOT.

**Recibe:** WoE congelados y lista seleccionada en TRAIN.

**Produce:** X_train_uncorr, X_test_uncorr, X_oot_uncorr.

**Funciones:** 4.0.0.multivariado.py: apply_selection.

In [ ]:
# MULT.11: Aplicación de variables finales a TEST/OOT
# Funciones: 4.0.0.multivariado.py: apply_selection
matrices_uncorr = mult.apply_selection(matrices_woe, variables_finales)
X_train_uncorr, X_test_uncorr, X_oot_uncorr = [matrices_uncorr[s] for s in mult.SEGMENTS]
for segmento, matriz in matrices_uncorr.items():
    print(segmento, matriz.shape)
display(X_train_uncorr.head())

## MULT.12 — Validación de matrices finales

**Qué hace:** Validación de matrices finales.

**Recibe:** Originales, targets, WoE, finales y correlación TRAIN.

**Produce:** df_validacion_final y comprobación de invariantes.

**Funciones:** 4.0.0.multivariado.py: validate_final; 3.0.0.bivariado.py: load_v4.

In [ ]:
# MULT.12: Validación de matrices finales
# Funciones: 4.0.0.multivariado.py: validate_final; 3.0.0.bivariado.py: load_v4
df_validacion_final = mult.validate_final(muestras_biv, matrices_woe, matrices_uncorr,
    corr_woe, variables_finales, CORR_THRESHOLD)
for segmento in mult.SEGMENTS:
    pd.testing.assert_series_equal(targets_originales[segmento], muestras_biv['y'][segmento], check_exact=True)
originales_releidos = biv.load_v4()
for segmento in mult.SEGMENTS:
    pd.testing.assert_frame_equal(muestras_biv['X'][segmento], originales_releidos['X'][segmento], check_exact=True)
assert mult.sha256(BINNING_PATH) == binning_hash_inicial
display(df_validacion_final)

## MULT.13 — Exportación de artefactos

**Qué hace:** Exportación de artefactos.

**Recibe:** Matrices validadas, reportes y figuras en memoria.

**Produce:** CSV, JSON de selección, manifiesto de filas, PNG y reconstrucción exacta.

**Funciones:** 4.0.0.multivariado.py: export_results, load_selection, reconstruct.

In [ ]:
# MULT.13: Exportación de artefactos
# Funciones: 4.0.0.multivariado.py: export_results, load_selection, reconstruct
seleccion_correlacion = mult.export_results(MULT_ROOT, muestras_biv, matrices_uncorr,
    df_iv_final, corr_woe, df_correlacion_decisiones, df_correlaciones_descartadas,
    variables_finales, CORR_THRESHOLD, BINNING_PATH,
    {'heatmap_completo':fig_heatmap_completo, 'heatmap_final':fig_heatmap_final, 'ranking_iv':fig_ranking_iv})
seleccion_recuperada = mult.load_selection(MULT_ROOT / 'seleccion_correlacion.json', BINNING_PATH)
reconstruidas = mult.reconstruct(muestras_biv, seleccion_recuperada, biv, BINNING_PATH)
for segmento in mult.SEGMENTS:
    pd.testing.assert_frame_equal(matrices_uncorr[segmento], reconstruidas[segmento], check_exact=True)
df_validacion_entradas.to_csv(MULT_ROOT / 'evidencias/validacion_entradas.csv', index=False)
df_validacion_final.to_csv(MULT_ROOT / 'evidencias/validacion_matrices.csv', index=False)
df_diagnosticos_woe.to_csv(MULT_ROOT / 'evidencias/diagnosticos_transformacion.csv', index=False)
print('Exportado y reconstruido exactamente sin reaprender correlaciones:', MULT_ROOT)

## MULT.14 — Resumen final para modelamiento

**Qué hace:** Resumen final para modelamiento.

**Recibe:** Selección y matrices finales.

**Produce:** Resumen para la próxima fase; sin ajustar modelo.

**Funciones:** 4.0.0.multivariado.py: resultados validados.

In [ ]:
# MULT.14: Resumen final para modelamiento
# Funciones: 4.0.0.multivariado.py: resultados validados
display(df_correlacion_decisiones.loc[df_correlacion_decisiones.Estado_correlacion.eq('CONSERVADA'),
    ['Variable','Ranking_IV','IV_final']])
print(f'B.3: {len(cols_por_iv)} aprobadas → {len(variables_finales)} finalistas; '
      f'{len(cols_por_iv)-len(variables_finales)} eliminadas por correlación.')
print('Selección exclusivamente TRAIN; Pearson absoluto >', CORR_THRESHOLD)
print('WoE congelado, targets e índices intactos. No se ha entrenado regresión logística.')
pruebas_path = MULT_ROOT / 'evidencias/pruebas.json'
if pruebas_path.exists():
    display(pd.DataFrame([json.loads(pruebas_path.read_text(encoding='utf-8'))]))